# 01 — Khám phá dữ liệu & tiền xử lý

**Chủ: Huy** · lý thuyết đi kèm: `docs/LY_THUYET.md` phần 1

Notebook này trả lời 4 câu, theo thứ tự:

1. Dữ liệu trông thế nào? (43 lớp, mất cân bằng bao nhiêu, ảnh gốc to nhỏ cỡ nào)
2. Vì sao cần cân bằng sáng? (so sánh trực tiếp 4 chế độ)
3. ★ Vì sao **không** được split random? (cấu trúc track — phần quan trọng nhất)
4. Augmentation nào được dùng, và **hai phép bị cấm** vì sao bị cấm?

> Chạy `python scripts/prepare_data.py --download` trước nếu chưa có dữ liệu.

In [ ]:
# Cho notebook thấy src/.  Chạy ô này đầu tiên.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)          # mọi đường dẫn trong repo là tương đối từ gốc

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from gtsrb import CLASS_NAMES, NUM_CLASSES
print("torch", torch.__version__, "| MPS", torch.backends.mps.is_available(),
      "| CUDA", torch.cuda.is_available())

## 1. Dữ liệu trông thế nào

`index.csv` là **nguồn sự thật duy nhất** về dữ liệu cho cả nhóm — không ai đọc ảnh
trực tiếp từ thư mục. Cột `track_id` là khoá chống rò rỉ, xem mục 3.

In [ ]:
index = pd.read_csv("data/processed/index.csv")
print("Tổng số ảnh:", len(index))
print(index.groupby("source").size().to_string())
print()
print("Phân bố split:")
print(index["split"].value_counts().to_string())
index.head(3)

In [ ]:
# Mất cân bằng lớp
counts = index[index["source"] == "train"]["class_id"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(counts.index, counts.values, color="steelblue")
ax.set_xlabel("class_id"); ax.set_ylabel("số ảnh train")
ax.set_title(f"Phân bố 43 lớp — mất cân bằng {counts.max()/counts.min():.1f}:1")
ax.axhline(counts.mean(), ls="--", c="red", label=f"trung bình {counts.mean():.0f}")
ax.legend(); plt.tight_layout(); plt.show()

print(f"Nhiều nhất : lớp {counts.idxmax()} ({CLASS_NAMES[counts.idxmax()]}) = {counts.max()} ảnh")
print(f"Ít nhất    : lớp {counts.idxmin()} ({CLASS_NAMES[counts.idxmin()]}) = {counts.min()} ảnh")
print(f"Tỉ lệ      : {counts.max()/counts.min():.1f}:1")

**Nhận xét.** Mất cân bằng khoảng 10,7:1. Nhóm **không** resample, vì:

- phân phối này phản ánh tần suất biển báo thật trên đường Đức;
- nhóm can thiệp ở **chỉ số** (dùng macro-F1 thay accuracy) thay vì ở **dữ liệu** —
  đó là chỗ can thiệp đúng hơn;
- vẫn stratify theo lớp khi split nên tỉ lệ train/val/test giống nhau.

Lý do đầy đủ: `docs/LY_THUYET.md` mục 1.4.

In [ ]:
# Kích thước ảnh gốc — giải thích vì sao resize là bắt buộc
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(index["width"], bins=60, color="steelblue")
axes[0].set_title("Chiều rộng ảnh gốc"); axes[0].set_xlabel("pixel")
axes[1].scatter(index["width"], index["height"], s=1, alpha=0.2)
axes[1].set_title("Rộng vs Cao"); axes[1].set_xlabel("rộng"); axes[1].set_ylabel("cao")
plt.tight_layout(); plt.show()

print(index[["width", "height"]].describe().loc[["min", "50%", "max"]].to_string())

**Nhận xét quan trọng cho phần M3.** Trung vị kích thước ảnh gốc chỉ khoảng
**50×50 pixel**. Nghĩa là khi M3 upsample lên 224×224, **không có thêm chi tiết thật nào** —
nội suy không tạo ra thông tin. Upsample chỉ để khớp **scale và stride** của mạng pretrained
(ba backbone downsample 32 lần, 48/32 = 1,5 là vô dụng). Đây là điều phải nói rõ khi
trình bày, đừng để giảng viên hỏi mới nói.

## 2. Vì sao cần cân bằng sáng — so sánh 4 chế độ

- `none` — RGB thô
- `he_gray` — ảnh xám + `equalizeHist` (cách của Sermanet & LeCun 2011)
- `he_y` — equalize kênh Y của YUV, **giữ màu**
- `clahe` — CLAHE trên kênh L của LAB ← **mặc định của nhóm**

In [ ]:
from gtsrb.data.preprocess import PREPROCESS_MODES, load_and_process

# Chọn vài ảnh TỐI để thấy rõ tác dụng (ảnh đã sáng thì equalize không khác nhiều)
import cv2
sample = index[index["source"] == "train"].sample(200, random_state=0)
brightness = []
for row in sample.itertuples(index=False):
    img = cv2.imread(row.path, cv2.IMREAD_GRAYSCALE)
    brightness.append(img.mean() if img is not None else 255)
sample = sample.assign(bright=brightness).nsmallest(5, "bright")

fig, axes = plt.subplots(len(PREPROCESS_MODES), len(sample),
                         figsize=(2.1 * len(sample), 2.1 * len(PREPROCESS_MODES)))
for col, row in enumerate(sample.itertuples(index=False)):
    roi = (row.roi_x1, row.roi_y1, row.roi_x2, row.roi_y2)
    for r, mode in enumerate(PREPROCESS_MODES):
        axes[r][col].imshow(load_and_process(row.path, roi, 96, mode))
        axes[r][col].set_xticks([]); axes[r][col].set_yticks([])
        if col == 0:
            axes[r][col].set_ylabel(mode, fontsize=11)
        if r == 0:
            axes[r][col].set_title(f"độ sáng {row.bright:.0f}", fontsize=8)
fig.suptitle("5 ảnh TỐI nhất qua 4 chế độ cân bằng sáng", fontsize=12)
plt.tight_layout(); plt.savefig("reports/figures/preprocess_compare.png", dpi=130); plt.show()

**Nhận xét.**

- `he_gray` làm rõ tương phản nhưng **mất màu** — mà màu biển báo mang nghĩa
  (đỏ = cấm, xanh = bắt buộc).
- `he_y` giữ màu, nhưng vì dùng **một** CDF cho cả ảnh nên vùng phẳng bị kéo nhiễu lên mạnh.
- `clahe` làm theo từng ô 8×8 và **chặn trần** histogram (`clipLimit=2.0`) nên không
  khuếch đại nhiễu vô hạn → đây là lý do nhóm chọn nó làm mặc định.

Trục ablation số 3 sẽ **đo bằng số** xem lựa chọn này đáng mấy điểm macro-F1:
`python scripts/run_ablation.py --axes preprocess`.

## 3. ★ Vì sao KHÔNG được split random — cấu trúc track

GTSRB **không** chụp mỗi biển báo một lần. Xe chạy tới gần một tấm biển và camera quay
**30 frame liên tiếp** của *cùng tấm biển vật lý đó*. Tên file mã hoá điều này:
`000{track}_000{frame}.ppm`.

Ô dưới đây cho thấy trực tiếp: các frame trong một track gần như là bản sao của nhau.

In [ ]:
# Một track = 30 frame của CÙNG MỘT tấm biển vật lý
track = index[(index.source == "train") & (index.class_id == 2) & (index.track_id == 5)]
track = track.sort_values("frame_id")
print(f"Track này có {len(track)} frame của cùng một tấm biển lớp 2 ({CLASS_NAMES[2]})")

show = track.iloc[::4].head(8)
fig, axes = plt.subplots(1, len(show), figsize=(2 * len(show), 2.3))
for ax, row in zip(axes, show.itertuples(index=False)):
    roi = (row.roi_x1, row.roi_y1, row.roi_x2, row.roi_y2)
    ax.imshow(load_and_process(row.path, roi, 96, "clahe"))
    ax.set_title(f"frame {row.frame_id}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Nếu split RANDOM, các ảnh này nằm cả ở train VÀ val → model chỉ cần NHỚ",
             fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Đối chứng định lượng: split theo track vs split random
import shutil, tempfile
from gtsrb.data.split import check_leakage, make_split

print("CÁCH ĐÚNG — split theo track (index.csv hiện tại):")
print("  track dùng chung giữa train và val:", check_leakage(index)["n_shared_tracks"])

with tempfile.TemporaryDirectory() as tmp:
    tmp_csv = pathlib.Path(tmp) / "index.csv"
    shutil.copy("data/processed/index.csv", tmp_csv)
    make_split(tmp_csv, val_ratio=0.2, seed=42, group_by_track=False)
    leaky = check_leakage(pd.read_csv(tmp_csv))

print("\nCÁCH SAI — split random theo ảnh:")
print("  track dùng chung giữa train và val:", leaky["n_shared_tracks"])
print("\n→ Đây là bằng chứng mạnh nhất của nhóm. Để đo phần accuracy 'ảo':")
print("  make data-leaky && make train-m1   (rồi so với con số split đúng)")

**Dấu hiệu nhận biết có rò rỉ:** val cao hơn test một cách bất thường.
Split đúng thì val và test phải **gần nhau**, vì cả hai đều là "biển báo chưa từng thấy".

`tests/test_split.py` là **cổng chặn** của dự án: nó khẳng định giao tập track
giữa train và val là rỗng. Nếu test đó đỏ thì mọi số liệu của dự án vô giá trị.

## 4. Augmentation — và hai phép BỊ CẤM

Nhóm dùng: xoay ±15°, dịch ±10%, zoom 0,9–1,1, jitter độ sáng/tương phản ±0,2.

In [ ]:
from gtsrb.data.transforms import build_transform

row = index[(index.source == "train") & (index.class_id == 14)].iloc[0]
roi = (row.roi_x1, row.roi_y1, row.roi_x2, row.roi_y2)
base = load_and_process(row.path, roi, 48, "clahe")
tensor = torch.from_numpy(base).permute(2, 0, 1)

tf = build_transform("geo_photo", 48)
torch.manual_seed(0)
fig, axes = plt.subplots(1, 9, figsize=(16, 2.1))
axes[0].imshow(base); axes[0].set_title("gốc", fontsize=9)
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
for i, ax in enumerate(axes[1:]):
    ax.imshow(tf(tensor).permute(1, 2, 0).numpy())
    ax.set_title(f"aug {i+1}", fontsize=9)
fig.suptitle(f"8 biến thể augmentation của cùng 1 ảnh — {CLASS_NAMES[14]}", fontsize=11)
plt.tight_layout(); plt.savefig("reports/figures/aug_grid.png", dpi=130); plt.show()

In [ ]:
# ★ VÌ SAO CẤM horizontal flip — xem bằng mắt
from gtsrb import MIRROR_PAIRS

fig, axes = plt.subplots(2, len(MIRROR_PAIRS) * 2, figsize=(3 * len(MIRROR_PAIRS), 4.4))
for col, (a, b) in enumerate(MIRROR_PAIRS):
    for offset, cls in enumerate((a, b)):
        r = index[(index.source == "train") & (index.class_id == cls)].iloc[0]
        roi = (r.roi_x1, r.roi_y1, r.roi_x2, r.roi_y2)
        img = load_and_process(r.path, roi, 72, "clahe")
        j = col * 2 + offset
        axes[0][j].imshow(img)
        axes[0][j].set_title(f"lớp {cls}\n{CLASS_NAMES[cls][:20]}", fontsize=7.5)
        axes[1][j].imshow(img[:, ::-1])          # lật ngang
        axes[1][j].set_title(f"lật ngang\n→ giống lớp {b if cls==a else a}!",
                             fontsize=7.5, color="crimson")
for row_axes in axes:
    for ax in row_axes: ax.set_xticks([]); ax.set_yticks([])
axes[0][0].set_ylabel("gốc", fontsize=10); axes[1][0].set_ylabel("sau flip", fontsize=10)
fig.suptitle("CẤM RandomHorizontalFlip: 4 cặp lớp là ẢNH GƯƠNG của nhau → flip = SAI NHÃN",
             fontsize=12)
plt.tight_layout(); plt.savefig("reports/figures/why_no_flip.png", dpi=130); plt.show()
print("Các cặp gương:", MIRROR_PAIRS)

**Kết luận mục 4.** Hai phép bị cấm tuyệt đối:

| Phép | Vì sao cấm |
|---|---|
| `RandomHorizontalFlip` | 4 cặp lớp (33/34, 19/20, 36/37, 38/39) là ảnh gương của nhau → flip tạo dữ liệu **sai nhãn** |
| hue / saturation jitter | màu **mang nghĩa**: đỏ = cấm, xanh = bắt buộc. Đổi hue là đổi nghĩa biển báo |

Brightness/contrast thì **được**, vì đó đúng là biến thiên có thật của điều kiện chụp.

**Bài học khái quát:** augmentation phải tôn trọng **tính bất biến thật** của bài toán.
Dùng mù quáng "bộ augment tiêu chuẩn cho ảnh" là cách tự làm hỏng nhãn.

`tests/test_transforms.py` khoá luật này ở mức mã nguồn.